#CUAD Dataset Training

In [9]:
# Install GPU-compatible spaCy, transformers, and torch dependencies
!pip install -U spacy[cuda12x,transformers] torch
!pip install spacy-transformers

INFO: pip is looking at multiple versions of spacy[cuda12x,transformers] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of spacy[cuda12x,transformers] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.7 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create workspace directory inside Drive
PROJECT_DIR = "/content/drive/MyDrive/CUAD Converted Dataset"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
# Prevent PyTorch GPU memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"Current Working Directory: {os.getcwd()}")

Mounted at /content/drive
Current Working Directory: /content/drive/MyDrive/CUAD Converted Dataset


In [2]:
import torch
import spacy

print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    spacy.require_gpu()
    print("spaCy successfully configured to use GPU!")
else:
    print("WARNING: GPU not detected. Ensure runtime accelerator is set to T4 GPU.")

PyTorch CUDA Available: True
GPU Name: Tesla T4
spaCy successfully configured to use GPU!


In [3]:
!python -m spacy init config config.cfg \
  --lang en \
  --pipeline transformer,ner \
  --optimize accuracy \
  --gpu


✘ The provided output file already exists. To force overwriting the
config file, set the --force or -F flag.



In [4]:
with open("config.cfg", "r") as f:
    config = f.read()

# 1. Lower batch size for transformer component
config = config.replace('max_words = 2048', 'max_words = 512')

# 2. Add gradient accumulation to simulate larger batches while conserving VRAM
if '[components.transformer.model]' in config:
    config = config.replace(
        '[components.transformer.model]',
        '[components.transformer.model]\ngrad_factor = 2'
    )

with open("config.cfg", "w") as f:
    f.write(config)

print("config.cfg successfully updated for memory optimization!")

config.cfg successfully updated for memory optimization!


In [6]:
!pip install spacy-transformers
import spacy
from confection import Config

# 1. Regenerate a clean default config file
!python -m spacy init config config.cfg --lang en --pipeline transformer,ner --optimize accuracy --gpu --force

# 2. Load config via spaCy's native parser
config = Config().from_disk("config.cfg")

# 3. Update transformer model to Legal-BERT
config["components"]["transformer"]["model"]["name"] = "nlpaueb/legal-bert-base-uncased"
config["components"]["transformer"]["model"]["grad_factor"] = 2

# 4. Correctly configure GPU VRAM limits via batcher (max 1000 words per batch)
config["training"]["batcher"] = {
    "@batchers": "spacy.batch_by_words.v1",
    "size": {
        "@schedules": "compounding.v1",
        "start": 100,
        "stop": 1000,
        "compound": 1.001
    },
    "discard_oversize": True
}

# 5. Save corrected config
config.to_disk("config.cfg")

print("config.cfg correctly configured with valid spaCy batcher settings!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.9/769.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.5/313.5 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled

In [11]:
import spacy
from confection import Config

# Load your existing config file
config = Config().from_disk("config.cfg")

# Remove the unsupported grad_factor argument from TransformerModel.v3
if "grad_factor" in config["components"]["transformer"]["model"]:
    del config["components"]["transformer"]["model"]["grad_factor"]
    print("Successfully removed invalid 'grad_factor' key!")

# Save the corrected configuration back to disk
config.to_disk("config.cfg")

Successfully removed invalid 'grad_factor' key!


In [14]:
from confection import Config

# Load existing config
config = Config().from_disk("config.cfg")

# Set paths inside the configuration structure
config["corpora"]["train"]["path"] = "./train.spacy"
config["corpora"]["dev"]["path"] = "./val.spacy"

# Save updated config
config.to_disk("config.cfg")

print("Successfully configured dataset paths inside config.cfg!")

Successfully configured dataset paths inside config.cfg!


In [15]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [16]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --gpu-id 0

ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 340kB/s]
config.json: 1.02kB [00:00, 4.23MB/s]
vocab.txt: 222kB [00:00, 10.3MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 728kB/s]
2026-09-28 12:46:00.557461: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 162MB/s]
model.safetensors: 100% 440M/440M [00:07<00:00, 62.6MB/s]
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 

In [18]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("en")
doc_bin = DocBin().from_disk("./val.spacy")
docs = list(doc_bin.get_docs(nlp.vocab))

print(f"Total Validation Chunks: {len(docs)}")

# Inspect first 3 validation chunks
total_ents = 0
for idx, doc in enumerate(docs[:3]):
    print(f"\n--- Chunk {idx + 1} ({len(doc)} tokens) ---")
    print(f"Text Snippet: {doc.text[:100]}...")
    for ent in doc.ents:
        print(f"  Entity: {ent.text:<25} | Label: {ent.label_}")
        total_ents += 1

if total_ents == 0:
    print("\nWARNING: No entities found in validation sample! Re-check conversion mapping.")
else:
    print(f"\nValidation dataset successfully verified!")

Total Validation Chunks: 3554

--- Chunk 1 (235 tokens) ---
Text Snippet: Exhibit 10.14(a ) 

 SECOND AMENDED AND RESTATED   EXCLUSIVE AGENCY AND   MARKETING AGREEMENT by and...
  Entity: September 30 , 1998       | Label: EFFECTIVE_DATE

--- Chunk 2 (648 tokens) ---
Text Snippet: 23 

  Section 4.2 Steering Committee 23 

  Section 4.3 Business Units 25 

  Section 4.4 Global Su...
  Entity: Monsanto                  | Label: PARTY_ORG
  Entity: SECOND AMENDED AND RESTATED EXCLUSIVE AGENCY AND MARKETING AGREEMENT | Label: AGREEMENT
  Entity: Monsanto Company          | Label: PARTY_ORG
  Entity: The Scotts Company LLC    | Label: PARTY_ORG
  Entity: the " Agent "             | Label: PARTY_ORG

--- Chunk 3 (75 tokens) ---
Text Snippet: Other countries and territories included in the Original Agreement that , as of the Execution Date ,...

Validation dataset successfully verified!


In [19]:
from confection import Config

# Load config
config = Config().from_disk("config.cfg")

# 1. Lower learning rate for stable transformer fine-tuning
config["training"]["optimizer"]["learn_rate"]["initial_rate"] = 1e-05

# 2. Add gradient clipping to prevent loss spikes
config["training"]["optimizer"]["grad_clip"] = 1.0

# 3. Add warm-up steps to stabilize early backpropagation
config["training"]["optimizer"]["learn_rate"]["warmup_steps"] = 500

# 4. Save updated config
config.to_disk("config.cfg")

print("config.cfg updated with stabilized learning rate (1e-5) and gradient clipping!")

config.cfg updated with stabilized learning rate (1e-5) and gradient clipping!


In [20]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [21]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --gpu-id 0

ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2026-09-28 13:35:08.290847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toD

In [17]:
import spacy

# Load the best performing model artifact from disk
nlp = spacy.load("./contract_model/model-best")

test_contract_preamble = """
This MUTUAL NON-DISCLOSURE AGREEMENT is made and entered into as of October 15, 2024 (the "Effective Date"),
by and between Acme Global Solutions Inc., a Delaware corporation ("Disclosing Party"),
and Beta Enterprises LLC ("Receiving Party").
"""

doc = nlp(test_contract_preamble)

print("\n--- Extracted Entities ---")
for ent in doc.ents:
    print(f"Text: {ent.text:<35} | Label: {ent.label_}")

ValueError: [E002] Can't find factory for 'transformer' for language English (en). This usually happens when spaCy calls `nlp.create_pipe` with a custom component name that's not registered on the current language class. If you're using a custom component, make sure you've added the decorator `@Language.component` (for function components) or `@Language.factory` (for class components).

Available factories: merge_noun_chunks, merge_entities, merge_subtokens, en.lemmatizer